# Document Data Extraction

In [1]:
from dotenv import load_dotenv
import os

## Setup API Keys

In [2]:
load_dotenv()
SARVAM_API_KEY = os.getenv('SARVAM_API_KEY')

assert SARVAM_API_KEY is not None, 'Could NOT load SARVAM_API_KEY from .env'    

In [3]:
from sarvamai import SarvamAI
# from sarvamai.core.api_error import ApiError

In [4]:
client = SarvamAI(
    api_subscription_key=SARVAM_API_KEY,
)

## Extract Data from Document

In [5]:
def load_document(f_name: str):
    with open(f_name, 'rb') as f_in:
        return f_in.read()

In [6]:
data = load_document('./data/pdfs/letter.pdf')

### Define data elements that need to be extracted

In [7]:
schema = {
    "type": "object",
    "properties": {
        "name": {"type": "string", "description": "Full Name of sender"},
        "address": {"type": "string", "description": "Address of sender"},
        "phone_number": {"type": "string", "description": "Mobile number of sender"},
        "request_summary":  {"type": "string", "description": "Summary of Request"},
        "account_number":   {"type": "number", "description": "Account Number of sender (if present)"},
        "client_id":   {"type": "number", "description": "Client Id of sender (if present)"},
    },
}

### Submit Job

In [9]:
import json

In [11]:
job = client.doc_ai.extract(
    file=[('output', data, 'application/pdf')],
    language='en-IN',
    schema=json.dumps(schema),
    output_format='json'
)

In [12]:
job.job_id

'01a0c210-8f0a-7e5a-9367-d24695d1d356'

### Check Job Status

In [13]:
status = client.doc_ai.get_status(job_id=job.job_id)
status

DocAiJobStatusResponse(job_id='01a0c210-8f0a-7e5a-9367-d24695d1d356', status='completed', pipeline='extract', usage=DocAiUsage(pages_total=1, pages_processed=1, pages_succeeded=1, pages_failed=0), created_at='2026-09-21T03:44:20Z', updated_at='2026-09-21T03:44:26Z', $schema='https://api.sarvam.ai/schemas/PublicJobBody.json')

In [14]:
json.loads(status.json())

{'job_id': '01a0c210-8f0a-7e5a-9367-d24695d1d356',
 'status': 'completed',
 'pipeline': 'extract',
 'usage': {'pages_total': 1,
  'pages_processed': 1,
  'pages_succeeded': 1,
  'pages_failed': 0},
 'created_at': '2026-09-21T03:44:20Z',
 'updated_at': '2026-09-21T03:44:26Z',
 '$schema': 'https://api.sarvam.ai/schemas/PublicJobBody.json'}

### Get Extracted Data

In [20]:
extracted_data = client.doc_ai.get_results(job_id=job.job_id)
extracted_data.result

{'name': 'Amit Prasad',
 'address': 'Chennai - 600001',
 'phone_number': '9840011223',
 'request_summary': 'Closure of Account',
 'account_number': None,
 'client_id': None}